# 003 · functools

> **Standard Library › Functional Tools** · Python 3.14

`functools` holds tools that **take or return functions**: pre-filling arguments, folding a sequence, memoizing results, dispatching on argument type, and filling in comparison methods. Several of these replace things C# gives you through language features (overloading, `Lazy<T>`, `IComparer<T>`).

**In this notebook you will learn:**

- `partial` for pre-filling arguments, and `reduce` as Python's `Aggregate`
- memoization with `cache` / `lru_cache`, including `cache_info()`, `cache_clear()` and the hashable-arguments rule
- `cached_property` for lazily computed attributes
- `singledispatch` for "overload by type"
- `total_ordering`, `wraps` and `cmp_to_key`

## .NET mental model

| `functools` | Closest .NET idea |
|---|---|
| `partial(f, x)` | a lambda closing over an argument: `y => F(x, y)` |
| `reduce(f, xs, seed)` | LINQ `Aggregate(seed, f)` |
| `cache` / `lru_cache` | hand-written `Dictionary` memo, `ConcurrentDictionary.GetOrAdd`, or `MemoryCache` (Microsoft.Extensions.Caching.Memory) |
| `cached_property` | a `Lazy<T>` field behind a property |
| `singledispatch` | method overloading — but chosen at **runtime** from the first argument's type |
| `total_ordering` | implementing `IComparable<T>` and writing all the operators by hand |
| `cmp_to_key` | adapting a `Comparison<T>` / `IComparer<T>` |
| `wraps` | no equivalent — Python decorators replace functions, so metadata must be copied |

## `partial` — pre-fill some arguments

```csharp
Func<string, int> parseHex = s => Convert.ToInt32(s, 16);
```

`partial(func, *args, **kwargs)` returns a new callable with some arguments already supplied. It is equivalent to a small lambda, but it's introspectable (you can see `.func`, `.args`, `.keywords`).

In [ ]:
from functools import partial

parse_hex = partial(int, base=16)        # int(s, base=16)
print(parse_hex("ff"), parse_hex("1A"))  # 255 26

print(parse_hex.func, parse_hex.args, parse_hex.keywords)

Positional arguments are filled **from the left**. That's fine for `pow(2, ...)`, but awkward when you want to fix a later argument:

In [ ]:
power_of_two = partial(pow, 2)     # pow(2, exp)
print([power_of_two(n) for n in range(5)])

square = partial(pow, exp=2)       # keyword args fix any parameter that accepts them
print([square(n) for n in range(5)])

New in 3.14: `functools.Placeholder` reserves a positional slot, so you can fix a later positional argument even for functions that don't accept keywords (like `str.replace`):

In [ ]:
from functools import Placeholder

dash_to_underscore = partial(str.replace, Placeholder, "-", "_")   # str.replace(<s>, "-", "_")
print(dash_to_underscore("order-line-item"))

**Tip:** a lambda is just as good and often clearer. Prefer `partial` when you want a picklable callable (e.g. for `multiprocessing`) or want to fix keyword arguments cleanly.

## `reduce` — Aggregate

```csharp
var product = numbers.Aggregate(1, (acc, n) => acc * n);
var longest = words.Aggregate((a, b) => a.Length >= b.Length ? a : b);
```

`reduce(function, iterable[, initial])` folds left-to-right. Note the argument order: **function first**, seed **last**.

In [ ]:
from functools import reduce

numbers = [2, 3, 4]
print(reduce(lambda acc, n: acc * n, numbers, 1))       # 24  — Aggregate(1, ...)

words = ["kiwi", "banana", "fig", "cherry"]
print(reduce(lambda a, b: a if len(a) >= len(b) else b, words))  # 'banana'

⚠️ **Gotcha:** like `Aggregate()` without a seed, `reduce` on an empty sequence with no `initial` raises (C# throws `InvalidOperationException`). Pass a seed when the input might be empty.

In [ ]:
try:
    reduce(lambda a, b: a + b, [])
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

print(reduce(lambda a, b: a + b, [], 0))   # 0 — seed returned

**Tip:** `reduce` was deliberately moved out of the built-ins — for common folds there's a clearer built-in: `sum`, `max`, `min`, `any`, `all`, `math.prod`, `"".join`. Reach for `reduce` only when none fits (e.g. merging dicts, composing functions).

In [ ]:
import math

print(sum(numbers), math.prod(numbers), max(words, key=len))

configs = [{"debug": False}, {"port": 8080}, {"debug": True}]
print(reduce(lambda a, b: a | b, configs, {}))    # merge dicts, later wins

## `cache` and `lru_cache` — memoization

A decorator that remembers return values per argument tuple. In C# you'd write a `Dictionary<TArgs, TResult>` lookup by hand (or use `MemoryCache`).

- `@cache` — unbounded (3.9+). Simple and fastest.
- `@lru_cache(maxsize=128)` — keeps the `maxsize` most recently used results and evicts the rest.

In [ ]:
from functools import cache

@cache
def fib(n: int) -> int:
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(80))             # instant — without the cache this would take ages
print(fib.cache_info())    # hits, misses, maxsize (None = unbounded), currsize

`lru_cache` bounds memory. `cache_info()` lets you check how effective the cache is, and `cache_clear()` empties it (e.g. in tests, or when the underlying data changed).

In [ ]:
from functools import lru_cache

calls = 0

@lru_cache(maxsize=2)
def load_price(sku: str) -> float:
    global calls
    calls += 1                       # pretend this is a slow lookup
    return {"A1": 9.99, "B2": 4.50, "C3": 12.00}[sku]

for sku in ["A1", "A1", "B2", "A1", "C3", "B2"]:
    load_price(sku)
print(load_price.cache_info(), "| real lookups:", calls)

load_price.cache_clear()
print(load_price.cache_info())

⚠️ **Gotcha:** arguments are used as dictionary keys, so they must be **hashable**. Passing a `list` or `dict` raises `TypeError` — convert to a `tuple` / `frozenset` first.

In [ ]:
@cache
def total(items):
    return sum(items)

print(total((1, 2, 3)))           # tuple: fine
try:
    total([1, 2, 3])              # list: unhashable
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

More caching gotchas:

- Only cache **pure** functions — results are returned even if the outside world changed.
- Cached mutable return values are **shared**: if a caller mutates the returned list, every later caller sees it.
- `f(1)` and `f(x=1)` are cached as different keys.
- On methods, `@cache` holds a reference to `self`, keeping instances alive. Prefer `cached_property` (below) or a module-level function.

## `cached_property` — compute once, then store

```csharp
private readonly Lazy<Stats> _stats = new(() => ComputeStats());
public Stats Stats => _stats.Value;
```

`@cached_property` runs the method the first time the attribute is read, then stores the result in the instance's `__dict__`; later reads are plain attribute lookups.

In [ ]:
from functools import cached_property

class Report:
    def __init__(self, values):
        self.values = values

    @cached_property
    def summary(self):
        print("  (computing...)")
        return {"n": len(self.values), "total": sum(self.values)}

r = Report([4, 8, 15])
print(r.summary)
print(r.summary)     # no "(computing...)" the second time

Because the value is stored as a normal attribute, you invalidate it with `del`:

In [ ]:
r.values.append(100)
del r.summary          # forget the cached value
print(r.summary)       # recomputed

⚠️ **Gotcha:** `cached_property` needs a per-instance `__dict__`, so it doesn't work on classes that define `__slots__`. It's also writable (unlike `@property`) — assigning to it just overwrites the cached value.

## `singledispatch` — overload by type

C# picks an overload at **compile time** from the static types:

```csharp
static string Describe(int n)    => $"int {n}";
static string Describe(string s) => $"string of length {s.Length}";
static string Describe(object o) => "something else";
```

Python has no overloading — a second `def` just replaces the first. `@singledispatch` gives you a generic function that picks an implementation at **runtime** based on the type of the **first** argument. Register implementations with type annotations:

In [ ]:
from functools import singledispatch

@singledispatch
def describe(obj) -> str:              # fallback, like the `object` overload
    return f"something else: {obj!r}"

@describe.register
def _(n: int) -> str:
    return f"int {n}"

@describe.register
def _(s: str) -> str:
    return f"string of length {len(s)}"

@describe.register
def _(items: list | tuple) -> str:     # union types work (3.11+)
    return f"sequence of {len(items)} items"

for value in [42, "hello", [1, 2], (3, 4, 5), 3.5, True]:
    print(describe(value))

Note `True` dispatched to the `int` implementation — `bool` is a subclass of `int`, and dispatch follows the class hierarchy (most specific registered type wins). You can inspect the registry:

In [ ]:
print(describe.dispatch(bool) is describe.dispatch(int))   # True — bool uses the int version
print(sorted(t.__name__ for t in describe.registry))

**Tip:** for methods inside a class, use `functools.singledispatchmethod` — it dispatches on the first argument **after** `self`.

## `total_ordering` — write two comparisons, get all six

In C# you implement `IComparable<T>` and then still have to write `<`, `>`, `<=`, `>=` operators yourself. In Python, define `__eq__` plus **one** of `__lt__`, `__le__`, `__gt__`, `__ge__`, and `@total_ordering` fills in the rest *(see Syntax › Classes & OOP › 007 · Dunder Methods)*.

In [ ]:
from functools import total_ordering

@total_ordering
class Version:
    def __init__(self, major: int, minor: int):
        self.major, self.minor = major, minor

    def __eq__(self, other):
        return (self.major, self.minor) == (other.major, other.minor)

    def __lt__(self, other):
        return (self.major, self.minor) < (other.major, other.minor)

    def __repr__(self):
        return f"v{self.major}.{self.minor}"

a, b = Version(1, 4), Version(2, 0)
print(a < b, a <= b, a > b, a >= b)          # >, <=, >= were generated
print(sorted([b, a, Version(1, 10)]))

**Tip:** for plain data classes, `@dataclass(order=True)` generates all comparisons from the fields, with no extra decorator *(see Syntax › Classes & OOP › 008 · Dataclasses)*.

## `wraps` — keep a decorated function's identity

When a decorator returns a wrapper function, the wrapper's `__name__`, `__doc__` etc. replace the original's. `@wraps(func)` copies them across — always use it when writing decorators *(see Syntax › Functions › 006 · Decorators)*.

In [ ]:
from functools import wraps

def logged(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"calling {func.__name__}{args}")
        return func(*args, **kwargs)
    return wrapper

@logged
def add(a, b):
    """Add two numbers."""
    return a + b

print(add(2, 3))
print(add.__name__, "-", add.__doc__)   # 'add', not 'wrapper'

## `cmp_to_key` — use an old-style comparison function

```csharp
list.Sort((a, b) => a.Priority != b.Priority
    ? b.Priority.CompareTo(a.Priority)      // high priority first
    : string.Compare(a.Name, b.Name, StringComparison.Ordinal));
```

Python's `sorted`/`sort` take only a `key=` function, never a comparer. If you already have a C#-style comparison (`negative / 0 / positive`), `cmp_to_key` adapts it:

In [ ]:
from functools import cmp_to_key

tasks = [("write docs", 2), ("fix bug", 5), ("review", 2), ("deploy", 5)]

def compare(a, b):
    if a[1] != b[1]:
        return b[1] - a[1]                       # higher priority first
    return (a[0] > b[0]) - (a[0] < b[0])         # then name ascending (-1/0/1)

print(sorted(tasks, key=cmp_to_key(compare)))

**Tip:** usually a key function is simpler and faster. Here a tuple key with a negated number does the same job:

In [ ]:
print(sorted(tasks, key=lambda t: (-t[1], t[0])))

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Pre-fill arguments | `y => F(x, y)` | `partial(f, x)` |
| Fill a later positional arg | lambda | `partial(f, Placeholder, x)` (3.14+) |
| Fold / aggregate | `.Aggregate(seed, f)` | `reduce(f, xs, seed)` |
| Memoize | `Dictionary` / `MemoryCache` | `@cache` / `@lru_cache(maxsize=n)` |
| Cache statistics / reset | — | `f.cache_info()` / `f.cache_clear()` |
| Lazy attribute | `Lazy<T>` | `@cached_property` |
| Overload by type | method overloads | `@singledispatch` + `.register` |
| All comparison operators | `IComparable<T>` + operators | `__eq__` + `__lt__` + `@total_ordering` |
| Keep metadata in decorators | — | `@wraps(func)` |
| Sort with a comparer | `Sort(Comparison<T>)` | `sorted(xs, key=cmp_to_key(cmp))` |

## Exercises

1. Use `partial` to create `int_from_binary` that parses strings like `"1011"`, then convert `["1", "10", "1011"]`.
2. Write a memoized function `ways(n)` that counts how many ways you can climb `n` stairs taking 1 or 2 steps at a time. Print `ways(50)` and the cache statistics.
3. Write a `@singledispatch` function `to_json_value` that returns numbers unchanged, quotes strings (`'"text"'`), turns `None` into `"null"`, and joins lists recursively like `"[1, \"a\"]"`.
4. Use `reduce` to compose a list of functions `[str.strip, str.lower, str.title]` into a single function and apply it to `"  hELLO wORLD "`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from functools import partial, cache, singledispatch, reduce

# 1
int_from_binary = partial(int, base=2)
print([int_from_binary(s) for s in ["1", "10", "1011"]])   # [1, 2, 11]

# 2
@cache
def ways(n: int) -> int:
    return 1 if n <= 1 else ways(n - 1) + ways(n - 2)

print(ways(50), ways.cache_info())

# 3 — register by type; NoneType is type(None)
@singledispatch
def to_json_value(value):
    raise TypeError(f"unsupported: {type(value).__name__}")

@to_json_value.register
def _(value: int | float):
    return value

@to_json_value.register
def _(value: str):
    return f'"{value}"'

@to_json_value.register(type(None))    # explicit type instead of an annotation
def _(value):
    return "null"

@to_json_value.register
def _(value: list):
    return "[" + ", ".join(str(to_json_value(v)) for v in value) + "]"

print(to_json_value([1, "a", None, [2.5]]))   # [1, "a", null, [2.5]]

# 4 — fold the functions into one pipeline
steps = [str.strip, str.lower, str.title]
pipeline = reduce(lambda f, g: lambda s: g(f(s)), steps)
print(repr(pipeline("  hELLO wORLD ")))   # 'Hello World'
```

</details>